# 04: モデル監視とデータドリフト検知

## このNotebookの狙い
**コードを変えていないのにモデルの精度が落ちる**現象を理解し、それを統計的に検知する仕組みを実装します。

### 用語整理
| 用語 | 意味 | 例 |
| --- | --- | --- |
| **Data Drift（特徴量ドリフト）** | 入力 X の分布が変わる | 新しい年代の顧客が増えた |
| **Concept Drift** | 入力と出力の関係 P(y\|X) が変わる | コロナで購買行動の意味が変わった |
| **Label Drift** | 出力 y の分布が変わる | スパムの割合が増えた |

### 第1回からの繋がり
ここで使う統計用語は第1回 EDA で触れたもの：
- **確率変数・確率分布**
- **記述統計**（平均・分散・分位数）
- **仮説検定**（KS検定・χ²検定）

「教科書で習う統計」が「本番運用で精度劣化を検知する道具」として活きる、という話です。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語フォントの設定（グラフの日本語ラベルの文字化け防止）
from scipy import stats      # KS検定・χ²検定を使う

rng = np.random.default_rng(42)  # 乱数生成器（シード固定で結果を再現可能にする）

## 1. ドリフトをシミュレート

学習時のデータ（reference）と、本番運用後のデータ（current）を擬似的に作ります。
本番側は分布が**少しずれた**設定にします。

In [ ]:
n = 5000
# 学習時: 平均 50, 標準偏差 10 の正規分布
reference = rng.normal(loc=50, scale=10, size=n)
# 本番: 平均が 55 にズレ、ばらつきも少し大きい
current = rng.normal(loc=55, scale=12, size=n)

df = pd.DataFrame({'reference': reference, 'current': current})
df.describe()  # 平均・標準偏差・分位数を並べて比較（記述統計）

In [ ]:
# ヒストグラムで重ねて可視化
fig, ax = plt.subplots(figsize=(8, 4))
# density=True: 件数ではなく確率密度で描く（面積の合計が1になり、形を比べやすい）
# alpha=0.5: 半透明にして重なりを見えるようにする
ax.hist(reference, bins=40, alpha=0.5, label='reference (学習時)', density=True)
ax.hist(current, bins=40, alpha=0.5, label='current (本番)', density=True)
ax.set_title('特徴量分布のズレ（Data Drift）')
ax.set_xlabel('feature value')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
plt.show()

## 2. KS検定（Kolmogorov-Smirnov 検定）

**2つの分布が同じか**を検定する有名な手法。第1回の「仮説検定」がここで活きる。

- 帰無仮説 H0: 2分布は同じ
- p値 < 0.05 なら H0 棄却 ＝「分布は異なる」と判定
- 統計量 D は「2つの累積分布関数の最大乖離」

In [ ]:
# 2標本KS検定: 2つのサンプルが同じ分布から来たかを検定する
ks_stat, p_value = stats.ks_2samp(reference, current)
print(f'KS統計量 D = {ks_stat:.4f}')
print(f'p値          = {p_value:.4e}')
if p_value < 0.05:                # 有意水準 5%
    print('→ 帰無仮説を棄却。分布は異なる（ドリフトあり）')
else:
    print('→ ドリフトの証拠なし')

## 3. PSI（Population Stability Index）

実務でよく使われるドリフト指標。**ヒストグラムの bin ごとの割合差**を集計する。

$$\mathrm{PSI} = \sum_i (p_i^{\text{cur}} - p_i^{\text{ref}}) \cdot \ln \frac{p_i^{\text{cur}}}{p_i^{\text{ref}}}$$

経験的しきい値：
- PSI < 0.1 : 安定
- 0.1 ≦ PSI < 0.2 : 注意
- PSI ≥ 0.2 : 大きなシフト → 再学習を検討

In [ ]:
def calc_psi(reference: np.ndarray, current: np.ndarray, bins: int = 10, eps: float = 1e-6) -> float:
    """PSI を計算する。

    1. reference の分位数で bin の境界を決める（等頻度ビン）
    2. 各 bin の reference / current の割合を求める
    3. (p_cur - p_ref) * ln(p_cur / p_ref) を bin ごとに合計
    """
    quantiles = np.linspace(0, 1, bins + 1)       # [0, 0.1, ..., 1.0]
    bin_edges = np.quantile(reference, quantiles)
    bin_edges[0] = -np.inf  # 端を伸ばして取りこぼし防止
    bin_edges[-1] = np.inf

    # 同じ bin 境界で、reference / current それぞれの件数を数える
    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)

    # 件数 → 割合に変換。eps を足すのは割合0のとき log(0) や 0除算を避けるため
    p_ref = ref_counts / ref_counts.sum() + eps
    p_cur = cur_counts / cur_counts.sum() + eps

    psi = float(np.sum((p_cur - p_ref) * np.log(p_cur / p_ref)))
    return psi

psi = calc_psi(reference, current, bins=10)
print(f'PSI = {psi:.4f}')
if psi < 0.1:
    print('→ 安定')
elif psi < 0.2:
    print('→ 注意（要観察）')
else:
    print('→ 大きなシフト。再学習を検討')

## 4. カテゴリ特徴量のドリフト：χ²検定

数値ではなくカテゴリ（性別・地域・カテゴリラベル等）の場合は **χ²（カイ二乗）検定**。
観測度数の表（クロス集計）と期待度数のズレで判定。

In [ ]:
# 例: 商品カテゴリ A/B/C/D の出現比が変化したかどうか
ref_counts = np.array([500, 300, 150, 50])   # 学習時
cur_counts = np.array([400, 250, 250, 100])  # 本番（C/Dが増加）

# 行=ソース（reference/current）、列=カテゴリ のクロス表
table = np.vstack([ref_counts, cur_counts])
# χ²独立性検定: 「どちらのデータか」と「カテゴリ」が無関係（＝分布が同じ）かを検定
# dof は自由度、expected は分布が同じだと仮定した場合の期待度数
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f'χ² = {chi2:.4f}, dof = {dof}, p値 = {p:.4e}')
if p < 0.05:
    print('→ カテゴリ分布に有意な変化あり')
else:
    print('→ 変化の証拠なし')

## 5. 検知 → アラート → 再学習トリガー

実運用では「検知して終わり」では意味がない。**しきい値超過時に何が起きるか**を設計する。

ここでは関数として組み立てる：
1. PSI を計算
2. しきい値超過なら通知（Slackやメール）の代わりに print
3. 連続 N 回続いたら **再学習ジョブをキック**するフラグを立てる

In [ ]:
from dataclasses import dataclass, field
from typing import List

# PSI を毎回記録し、しきい値超えが連続したら再学習を促す監視クラス
@dataclass
class DriftMonitor:
    threshold: float = 0.2                              # アラートを出す PSI のしきい値
    consecutive_required: int = 3                       # 何回連続で超えたら再学習とするか
    history: List[float] = field(default_factory=list)  # 過去の PSI の記録（インスタンスごとに別のリスト）

    def observe(self, ref: np.ndarray, cur: np.ndarray) -> dict:
        psi = calc_psi(ref, cur)
        self.history.append(psi)
        # 直近 consecutive_required 回分について、しきい値を超えたかを True/False で並べる
        over = [v >= self.threshold for v in self.history[-self.consecutive_required:]]
        # 記録が必要回数そろっていて、かつ全部超えていたら再学習
        should_retrain = len(over) == self.consecutive_required and all(over)
        result = {
            'psi': psi,
            'alert': psi >= self.threshold,
            'should_retrain': should_retrain,
            'history_tail': self.history[-self.consecutive_required:],
        }
        return result

monitor = DriftMonitor(threshold=0.2, consecutive_required=3)

# 4 日分のシミュレーション（だんだん本番側がズレていく）
# Day2〜4 の3日連続でしきい値を超えると、Day4 で retrain=True になる
for day, shift in enumerate([2, 4, 6, 8], 1):
    cur = rng.normal(loc=50 + shift, scale=10, size=n)  # 平均を shift だけずらした本番データ
    out = monitor.observe(reference, cur)
    print(f'Day {day}: shift={shift}, PSI={out["psi"]:.4f}, alert={out["alert"]}, retrain={out["should_retrain"]}')

## 6. 実運用での観点（コード以外）

### 何を監視するか
- **入力分布**（特徴量ごとに PSI / KS）
- **予測分布**（予測クラスの比率や信頼度のヒストグラム）
- **性能**（ラベルが手に入る場合は accuracy 等の時系列）
- **インフラ**（レイテンシ・エラー率・GPU使用率）

### 監視の難しさ：ラベル遅延
推論結果の正解（ラベル）は遅れて手に入る、または手に入らない場合も多い。
→ ラベルなしでも検知できる **入力分布監視（Data Drift）** が重要。

### 検知後のアクション
1. **記録**：時系列に PSI / KS の値を保存（Prometheus, BigQuery 等）
2. **可視化**：ダッシュボード（Grafana, Looker）
3. **アラート**：Slack / PagerDuty
4. **再学習トリガー**：CT パイプラインのキック

### 関連OSS
- **Evidently AI**: ドリフトレポートを HTML で生成
- **whylogs**: ログ収集ベースの統計プロファイル
- **NannyML**: ラベル無しでの性能推定

## まとめ

- ML 特有の運用問題＝**コード不変でモデルが腐る**
- 検知の道具：**KS検定（数値）／PSI（数値・実務標準）／χ²（カテゴリ）**
- これらは第1回の「**仮説検定・確率分布**」が本番で活きる例
- 検知だけでなく**何を起こすか（再学習）**を含めて設計

ここまでで、第8回の技術的なハンズオンは完了。次のスライドで全8回を振り返ります。